# Customer churn: does 99% mean new-customer generalization?

This public sample intentionally uses a row-wise split while retaining `customer_id`.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path('../public/customer_churn.csv')
df = pd.read_csv(data_path)
df[['customer_id', 'churned']].head()

Loaded 2880 rows across 480 customers


## Reported evaluation

The next cell uses a random row split. Repeated observations from one customer can land on both sides.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=['churned'])
y = df['churned']
categorical = [c for c in X.columns if X[c].dtype == 'object']
numeric = [c for c in X.columns if c not in categorical]
preprocess = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('numeric', Pipeline([('impute', SimpleImputer()), ('scale', StandardScaler())]), numeric),
])
model = Pipeline([('preprocess', preprocess), ('model', LogisticRegression(C=100, max_iter=2000))])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1729, stratify=y
)
model.fit(X_train, y_train)
probability = model.predict_proba(X_test)[:, 1]
prediction = (probability >= 0.5).astype(int)
print(f'Random row-split accuracy: {accuracy_score(y_test, prediction):.3f}')
print(f'Random row-split ROC AUC: {roc_auc_score(y_test, probability):.3f}')

Random row-split accuracy: 0.985
Random row-split ROC AUC: 0.984
Train/test customer overlap: 389 (100.0%)


COUNTERLAB_VERIFIED_RESULT={"chartData":[{"accuracy":0.984722222222,"rocAuc":0.983803512454,"runId":"random_row_split","sampleSize":720,"seed":1729,"splitStrategy":"random"},{"accuracy":0.594444444444,"rocAuc":0.641242588933,"runId":"customer_group_split","sampleSize":720,"seed":1729,"splitStrategy":"group"},{"accuracy":0.673611111111,"rocAuc":0.725006944659,"runId":"identity_ablation","sampleSize":720,"seed":1729,"splitStrategy":"random"}],"concept":"entity_leakage","fixture":{"customers":480,"rows":2880,"sha256":"5c482f39e4e948a92dab61bf9c9f5c6577fbe9fc688fd597c9fefd785ee1be70","targetRate":0.497569444444},"kernelVersion":"0.1.0","resultHash":"a6ae7652e04e4d70196f991c63b8f7bcb3b76f8c4ab833d3ce2b626df0ab6c94","runs":[{"dropFeatures":[],"entityCounts":{"test":389,"train":480},"entityOverlap":{"count":389,"rate":1.0},"featureSetFingerprint":"1fd4ea95c8cbe12d5598484f1907d86fc7587000ba5657e1faddd40cb1be4e44","groupBy":null,"id":"random_row_split","inputFingerprint":"5c482f39e4e948a92dab61

**Learner claim to test:** This result proves the model generalizes to customers it has never seen.